# Tutorial 01 — Blocks: circuits as composable objects

Everything the quantum computer *does* is a `Block`. There are two kinds:

* **`SimpleBlock`** — a flat gate list you fill with builder methods
  (`.h()`, `.cx()`, `.ry()`, ...).
* **`CompositeBlock`** — an ordered tree of child blocks, so circuits compose
  like Lego: `[state-prep, ansatz, readout]`.

On top of these, `qarp.blocks` ships a library of ready-made blocks
(`HnBlock`, `HEABlock`, `ComputationalBasisStateBlock`, `QFTBlock`,
`TrotterBlock`, ...) — see `mwe_blocks.ipynb` for the full catalogue.

## 1. SimpleBlock and the gate builders

In [ ]:
import numpy as np
from sympy import Symbol
from qarp.blocks import SimpleBlock

theta = Symbol("theta")

blk = SimpleBlock(3, name="demo")
blk.h(0)
blk.cx(0, 1)
blk.ry(2, theta)        # parametric gates accept sympy Symbols (or floats)
blk.rz(2, 0.25)         # angles are ALWAYS in radians
blk.build()
blk.plot(spacing=0.4)

### The `.build()` lifecycle

* **Before** `.build()`: add gates, compose, transform.
* **After** `.build()`: the block is finalised — `.symbols` is populated, and the
  block can be plotted, flattened, or handed to an engine. Calling `.build()`
  again is a no-op.

`.flatten()` returns the raw command stream (the C++ IR) — useful to see exactly
what will be compiled:

In [ ]:
print("symbols:", blk.symbols)
for cmd in blk.flatten():
    print(cmd)

> **Convention 1 — symbols are sorted.** `block.symbols` is sorted by symbol
> *name*, not by insertion order. When you zip parameter values to symbols,
> always go through `block.symbols`, never through your own gate order.

## 2. Composition

`CompositeBlock` glues child blocks in order. The canonical pattern is
`[state-preparation, ansatz, readout]`:

In [ ]:
from qarp.blocks import (
    CompositeBlock,
    ComputationalBasisStateBlock,
    HEABlock,
    ReadoutBlock,
)

prep = ComputationalBasisStateBlock([1, 1, 0, 0])   # X on qubits 0, 1
ansatz = HEABlock(4, 2, True, True, True, False)     # (n_qubits, n_layers, real, linear, circular, use_cz)
readout = ReadoutBlock(n_qubits=4)                   # measure all qubits, cbit q <- qubit q

circuit = CompositeBlock([prep, ansatz, readout]).build()
circuit.plot()

Each box is a child; pass `decompose_boxes=True` to `.plot()` to see the gates
inside.

### Measurement, three ways

* `block.measure(q, c)` (or a list of `(qubit, cbit)` pairs) — builder method on a
  block you're already filling, like in tutorial 00.
* `ReadoutBlock(n_qubits, qubits=..., cbits=...)` — a composable child for bulk
  readout, as above.
* `MeasureBlock(qubit, cbit)` — the single-measurement structural primitive both
  of the above reduce to.

For pure sampling the engine reports counts over all qubits even without
explicit measures, but making them explicit keeps the IR honest (and is
required for QASM/QIR export).

## 3. Setting parameter values

Symbolic parameters get values in one of two places:

* **At run time** — pass a `{symbol: value}` map to `engine.run(...)`
  (the usual way in optimisation loops; tutorials 03–04).
* **On the block** — `set_symbols` returns a **new** block with the values
  recorded (the original is untouched).

`set_symbols` is *lazy*: the substitution is stored on the new block and
applied when the block is flattened — which is exactly what the engine does
when it compiles. Bound symbols leave `bound.symbols` (it lists only the
symbols still free); the place to see the bound values is the command stream:

In [ ]:
bound = circuit.set_symbols({s: 0.1 for s in circuit.symbols})

print("original:", circuit.flatten()[2:5])   # Ry gates, still symbolic
print("bound:   ", bound.flatten()[2:5])     # same gates, values applied
print("bound.symbols lists only the free symbols:", bound.build().symbols)

> **Convention 2 — angles are radians.** Every rotation angle and every value
> you substitute for a symbol is interpreted in **radians**.
> `ry(0, np.pi)` flips `|0>` to `|1>`:

In [ ]:
from qarp.algorithms import Sampler
from qarp.engines import QarpEngine

flip = SimpleBlock(1)
flip.ry(0, np.pi)
flip.measure(0, 0)
flip.build()

s = Sampler(ket=flip, n_shots=1000)
engine = QarpEngine(seed=1)
engine.build([s])
engine.run()[0]

## 4. Bit order

> **Convention 3 — LSB-first.** Qubit `q` is bit `q`: in a sampled tuple,
> position `q` belongs to qubit `q`, and in an integer basis-state label, qubit
> `q` contributes `2**q`. Operator matrices (`op.sparse_matrix()`) use the same
> convention. `qarp.endianness` has the converters — use them only at a boundary with an
> external MSB-first tool (openfermion / cirq / pennylane):

In [ ]:
from qarp.endianness import bits_to_label, label_to_bits

bits = (0, 1, 1)            # qubit 0 = 0, qubit 1 = 1, qubit 2 = 1
print(bits_to_label(bits))  # 2**1 + 2**2 = 6
print(label_to_bits(6, 3))

## Poke at it

* `circuit.children` (composites) vs gate builders (simple blocks)
* `circuit.n_qubits`, `circuit.symbols`, `len(circuit.flatten())`
* `circuit.plot(decompose_boxes=True)`, `circuit.to_qasm3()`
* `block.dagger()` — the adjoint, applied lazily at `flatten()` time

**Next:** tutorial_02_primitives — turning circuits into numbers.